In [ ]:
# =========================
# [CELL 1] BUILD random-weight dataset (NO raw split)
# - Supports BOTH layouts:
#   (A) flat:   BASE_DIR/data_with_features_{split}.csv
#              BASE_DIR/data_with_features_{ALG}_{split}.csv
#   (B) folder: BASE_DIR/.../original/data_with_features_{split}.csv
#              BASE_DIR/.../{ALG}/data_with_features_{ALG}_{split}.csv
#
# - target fixed: label
# - HARD STOP if:
#   * any edge endpoint u/v contains "Unnamed"
#   * generated edge colname contains "Unnamed" or is index-like
# =========================

import os
import warnings
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import networkx as nx

warnings.filterwarnings("ignore")

# -------- Config --------
BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"
OUT_DATASET_DIR = os.path.join(BASE_DIR, "add_feature_dataset_random_weight")

DATA_PREFIX = "data_with_features"
TARGET_COL = "label"

GEXF_PATHS = {
    "GES":   os.path.join(BASE_DIR, "graph_GES.gexf"),
    "GOLEM": os.path.join(BASE_DIR, "graph_GOLEM.gexf"),
}

RAND_WEIGHT_SEED = 12345
STRICT_NODE_MATCH = True

# -------- HARD BLOCK helpers --------
def is_indexlike_col_name(c: str) -> bool:
    cl = str(c).strip().lower()
    return cl.startswith("unnamed") or cl in {"index", "_index"} or cl.endswith("_index")

def looks_like_index_series(s: pd.Series) -> bool:
    try:
        v = pd.to_numeric(s, errors="coerce")
        if v.isna().mean() > 0.3:
            return False
        n = len(v)
        if n <= 5:
            return False
        uniq_ratio = v.nunique(dropna=True) / float(n)
        if uniq_ratio < 0.98:
            return False
        vv = v.to_numpy()
        if np.all(vv == np.arange(n)): return True
        if np.all(vv == np.arange(1, n + 1)): return True
        if np.all(np.diff(vv) >= 0):
            dif = np.diff(vv)
            if np.mean(np.abs(dif - 1.0) < 1e-9) > 0.95:
                return True
    except Exception:
        return False
    return False

def drop_all_indexlike_cols(df: pd.DataFrame, name: str, aggressive_value_check: bool = True) -> pd.DataFrame:
    drop_cols = []
    for c in list(df.columns):
        if is_indexlike_col_name(c):
            drop_cols.append(c)

    if aggressive_value_check:
        for c in list(df.columns):
            if c in drop_cols:
                continue
            try:
                if looks_like_index_series(df[c]):
                    drop_cols.append(c)
            except Exception:
                pass

    if drop_cols:
        drop_cols = list(dict.fromkeys(drop_cols))
        print(f"[DROP] {name}: index-like columns removed -> {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).strip().lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed columns still exist after drop: {bad}")
    return df

def fatal_if_unnamed_in_edge_endpoint(u: str, v: str, alg: str):
    if "unnamed" in str(u).lower() or "unnamed" in str(v).lower():
        raise RuntimeError(f"[FATAL] {alg}: edge endpoint contains 'Unnamed' -> (u={u}, v={v})")

def fatal_if_bad_colname(col: str, context: str):
    if is_indexlike_col_name(col) or ("unnamed" in str(col).strip().lower()):
        raise RuntimeError(f"[FATAL] {context}: forbidden generated column name -> {col}")

def make_edge_col_name(alg: str, u: str, v: str) -> str:
    col = f"edge_{alg}__{u}__{v}"
    fatal_if_bad_colname(col, f"{alg}/edge_col_name")
    return col

def read_graph_edges_sorted(gexf_path: str) -> List[Tuple[str, str]]:
    if not os.path.exists(gexf_path):
        raise FileNotFoundError(f"[FATAL] GEXF not found: {gexf_path}")
    G = nx.read_gexf(gexf_path)
    edges = []
    if isinstance(G, (nx.MultiDiGraph, nx.MultiGraph)):
        for u, v, k in G.edges(keys=True):
            edges.append((str(u), str(v)))
    else:
        for u, v in G.edges():
            edges.append((str(u), str(v)))
    return sorted(edges, key=lambda x: (x[0], x[1]))

# -------- path resolver (flat OR folder) --------
def resolve_split_path(base_dir: str, dag: str, split: str) -> str:
    """
    dag:
      - "original" -> expects data_with_features_{split}.csv
      - "GES" etc  -> expects data_with_features_{dag}_{split}.csv
    """
    if dag == "original":
        fname = f"{DATA_PREFIX}_{split}.csv"
        flat = os.path.join(base_dir, fname)
        if os.path.exists(flat):
            return flat

        # folder candidates
        for root, dirs, files in os.walk(base_dir):
            if root.endswith(os.sep + "original") or os.path.basename(root) == "original":
                p = os.path.join(root, fname)
                if os.path.exists(p):
                    return p

        raise FileNotFoundError(
            f"[FATAL] original split not found for split={split}\n"
            f"Expected either:\n"
            f" - {flat}\n"
            f" - <somewhere under BASE_DIR>/original/{fname}\n"
        )
    else:
        fname = f"{DATA_PREFIX}_{dag}_{split}.csv"
        flat = os.path.join(base_dir, fname)
        if os.path.exists(flat):
            return flat

        # folder candidates
        for root, dirs, files in os.walk(base_dir):
            if os.path.basename(root) == dag:
                p = os.path.join(root, fname)
                if os.path.exists(p):
                    return p

        raise FileNotFoundError(
            f"[FATAL] {dag} split not found for split={split}\n"
            f"Expected either:\n"
            f" - {flat}\n"
            f" - <somewhere under BASE_DIR>/{dag}/{fname}\n"
        )

def load_split(dag: str, split: str) -> pd.DataFrame:
    p = resolve_split_path(BASE_DIR, dag, split)
    df = pd.read_csv(p, low_memory=False)
    df = drop_all_indexlike_cols(df, f"{dag}_{split}", aggressive_value_check=True)
    if TARGET_COL not in df.columns:
        raise RuntimeError(f"[FATAL] target '{TARGET_COL}' missing in {p}")
    return df

# -------- build random-weight dataset --------
def build_random_weight_dataset():
    # already-built quick check
    need = [
        os.path.join(OUT_DATASET_DIR, "original", f"{DATA_PREFIX}_train.csv"),
        os.path.join(OUT_DATASET_DIR, "original", f"{DATA_PREFIX}_val.csv"),
        os.path.join(OUT_DATASET_DIR, "original", f"{DATA_PREFIX}_test.csv"),
        os.path.join(OUT_DATASET_DIR, "GES",   f"{DATA_PREFIX}_GES_train.csv"),
        os.path.join(OUT_DATASET_DIR, "GES",   f"{DATA_PREFIX}_GES_val.csv"),
        os.path.join(OUT_DATASET_DIR, "GES",   f"{DATA_PREFIX}_GES_test.csv"),
        os.path.join(OUT_DATASET_DIR, "GOLEM", f"{DATA_PREFIX}_GOLEM_train.csv"),
        os.path.join(OUT_DATASET_DIR, "GOLEM", f"{DATA_PREFIX}_GOLEM_val.csv"),
        os.path.join(OUT_DATASET_DIR, "GOLEM", f"{DATA_PREFIX}_GOLEM_test.csv"),
    ]
    if all(os.path.exists(p) for p in need):
        print(f"[SKIP] random-weight dataset already exists: {OUT_DATASET_DIR}")
        return

    print("[LOAD] existing splits (NO raw split creation)")
    df_train = load_split("original", "train")
    df_val   = load_split("original", "val")
    df_test  = load_split("original", "test")

    # base cols = not target, not edge_*
    base_cols = [c for c in df_train.columns if c != TARGET_COL and not str(c).startswith("edge_")]
    bad_base = [c for c in base_cols if is_indexlike_col_name(c) or ("unnamed" in str(c).lower())]
    if bad_base:
        raise RuntimeError(f"[FATAL] base columns contain forbidden names: {bad_base[:30]} (total {len(bad_base)})")

    # save original (base + label)
    os.makedirs(os.path.join(OUT_DATASET_DIR, "original"), exist_ok=True)
    for split, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
        out = df[base_cols + [TARGET_COL]].copy()
        out.to_csv(os.path.join(OUT_DATASET_DIR, "original", f"{DATA_PREFIX}_{split}.csv"), index=False)

    rng = np.random.default_rng(RAND_WEIGHT_SEED)

    for alg in ["GES", "GOLEM"]:
        df_alg_train = load_split(alg, "train")
        df_alg_val   = load_split(alg, "val")
        df_alg_test  = load_split(alg, "test")

        # IMPORTANT: we still build edges from base_cols and compute X[u]*X[v]
        edges = read_graph_edges_sorted(GEXF_PATHS[alg])

        for (u, v) in edges:
            fatal_if_unnamed_in_edge_endpoint(u, v, alg)

        w = rng.uniform(-1.0, 1.0, size=len(edges)).astype(np.float32)

        missing = [(u, v) for (u, v) in edges if (u not in base_cols or v not in base_cols)]
        if missing and STRICT_NODE_MATCH:
            raise RuntimeError(
                f"[FATAL] {alg}: some edge endpoints are missing from base columns.\n"
                f"Examples: {missing[:20]}"
            )

        os.makedirs(os.path.join(OUT_DATASET_DIR, alg), exist_ok=True)

        def add_edges(df_orig_like: pd.DataFrame) -> pd.DataFrame:
            out = df_orig_like[base_cols + [TARGET_COL]].copy()
            X = out[base_cols]

            for (u, v), ww in zip(edges, w):
                if u not in X.columns or v not in X.columns:
                    continue
                col = make_edge_col_name(alg, u, v)
                out[col] = (ww * (X[u].to_numpy(np.float32) * X[v].to_numpy(np.float32))).astype(np.float32)
            return out

        tr = add_edges(df_train)
        va = add_edges(df_val)
        te = add_edges(df_test)

        tr.to_csv(os.path.join(OUT_DATASET_DIR, alg, f"{DATA_PREFIX}_{alg}_train.csv"), index=False)
        va.to_csv(os.path.join(OUT_DATASET_DIR, alg, f"{DATA_PREFIX}_{alg}_val.csv"), index=False)
        te.to_csv(os.path.join(OUT_DATASET_DIR, alg, f"{DATA_PREFIX}_{alg}_test.csv"), index=False)

        print(f"[OK] Saved {alg} random splits -> {os.path.join(OUT_DATASET_DIR, alg)}")

    print(f"[DONE] Built random-weight dataset at: {OUT_DATASET_DIR}")

# Execute
build_random_weight_dataset()


In [ ]:
# =========================
# [CELL 2] RUN TOP2 repeats (F + OF only) on random-weight dataset
#
# - Assumes random-weight dataset already exists at:
#     BASE_DIR/add_feature_dataset_random_weight/
#
# - target column fixed: label
# - feature lists MUST be loaded from features_used.csv (fixed)
# - saves repeat-level results ONLY (no per-sample predictions)
# - GPU try:
#   * XGBoost: gpu_hist (fallback CPU)
#   * LightGBM: device_type=gpu try (fallback CPU)
# =========================

import os
import json
import time
import hashlib
import warnings
from typing import Dict, List

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")

# -------- Config --------
BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"
DATASET_DIR = os.path.join(BASE_DIR, "add_feature_dataset_random_weight")
DATA_PREFIX = "data_with_features"
TARGET_COL = "label"

# repeats
N_REPEATS = 100
BASE_SEED = 42
THR_GRID = 101
ECE_BINS = 15

# TOP2 configs (feature list comes from features_used.csv)
CFG_F  = dict(SET="F",  DAG="GES",   MODEL="LightGBM", K_EDGE=44, N_FEAT=44)
CFG_OF = dict(SET="OF", DAG="GOLEM", MODEL="XGBoost",  K_EDGE=8,  N_FEAT=21)

# features_used.csv path (Drive에 두는 것을 권장)
FEATURES_USED_PATH_CANDIDATES = [
    os.path.join(BASE_DIR, "features_used.csv"),
    "/mnt/data/features_used.csv",  # (현재 대화 업로드 파일이 이 경로로 들어오는 경우 대비)
]

OUT_DIR = os.path.join(BASE_DIR, "results_tables_random_weight_top3")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_RESULTS  = os.path.join(OUT_DIR, "results_top2_repeats.csv")
OUT_FEATURES = os.path.join(OUT_DIR, "features_used_top2_repeats.csv")


# -------- HARD BLOCK helpers --------
def is_indexlike_col_name(c: str) -> bool:
    cl = str(c).strip().lower()
    return cl.startswith("unnamed") or cl in {"index", "_index"} or cl.endswith("_index")

def looks_like_index_series(s: pd.Series) -> bool:
    try:
        v = pd.to_numeric(s, errors="coerce")
        if v.isna().mean() > 0.3:
            return False
        n = len(v)
        if n <= 5:
            return False
        uniq_ratio = v.nunique(dropna=True) / float(n)
        if uniq_ratio < 0.98:
            return False
        vv = v.to_numpy()
        if np.all(vv == np.arange(n)): return True
        if np.all(vv == np.arange(1, n + 1)): return True
        if np.all(np.diff(vv) >= 0):
            dif = np.diff(vv)
            if np.mean(np.abs(dif - 1.0) < 1e-9) > 0.95:
                return True
    except Exception:
        return False
    return False

def drop_all_indexlike_cols(df: pd.DataFrame, name: str, aggressive_value_check: bool = True) -> pd.DataFrame:
    drop_cols = []
    for c in list(df.columns):
        if is_indexlike_col_name(c):
            drop_cols.append(c)

    if aggressive_value_check:
        for c in list(df.columns):
            if c in drop_cols:
                continue
            try:
                if looks_like_index_series(df[c]):
                    drop_cols.append(c)
            except Exception:
                pass

    if drop_cols:
        drop_cols = list(dict.fromkeys(drop_cols))
        print(f"[DROP] {name}: index-like columns removed -> {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).strip().lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed columns still exist after drop: {bad}")
    return df

def filter_out_index_cols(cols: List[str]) -> List[str]:
    out = []
    for c in cols:
        if is_indexlike_col_name(c):
            continue
        if str(c).strip().lower().startswith("unnamed"):
            continue
        out.append(c)
    return out

def hard_check_feature_list(cols: List[str], context: str) -> List[str]:
    cols = filter_out_index_cols(cols)
    bad = [c for c in cols if (is_indexlike_col_name(c) or ("unnamed" in str(c).strip().lower()))]
    if bad:
        raise RuntimeError(f"[FATAL] {context}: forbidden columns in feature list: {bad[:30]} (total {len(bad)})")
    return cols


# -------- metrics --------
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    y_true = y_true.astype(int)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        acc = float(y_true[mask].mean())
        conf = float(y_prob[mask].mean())
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.astype(float)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    return float(np.mean((y_prob - y_true) ** 2))

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Brier": brier_score(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_BINS),
    }


# -------- IO utils --------
def ensure_cols(df: pd.DataFrame, cols: List[str], name: str):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise RuntimeError(f"[FATAL] {name}: missing columns: {miss[:30]} (total {len(miss)})")

def load_split(dag: str, split: str) -> pd.DataFrame:
    if dag == "original":
        path = os.path.join(DATASET_DIR, "original", f"{DATA_PREFIX}_{split}.csv")
    else:
        path = os.path.join(DATASET_DIR, dag, f"{DATA_PREFIX}_{dag}_{split}.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"[FATAL] Missing dataset: {path}")
    df = pd.read_csv(path, low_memory=False)
    df = drop_all_indexlike_cols(df, f"{dag}_{split}", aggressive_value_check=True)
    if TARGET_COL not in df.columns:
        raise RuntimeError(f"[FATAL] target '{TARGET_COL}' missing in {path}")
    return df

def find_features_used_path() -> str:
    for p in FEATURES_USED_PATH_CANDIDATES:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"[FATAL] features_used.csv not found. Tried: {FEATURES_USED_PATH_CANDIDATES}")

def load_fixed_feature_list(df_used: pd.DataFrame, set_name: str, dag: str, k_edge) -> List[str]:
    need = {"SET", "DAG", "FEATURES_JSON"}
    miss = [c for c in need if c not in df_used.columns]
    if miss:
        raise RuntimeError(f"[FATAL] features_used.csv missing columns: {miss}")

    m = (df_used["SET"].astype(str) == str(set_name)) & (df_used["DAG"].astype(str) == str(dag))
    if "K_EDGE" in df_used.columns:
        m = m & (df_used["K_EDGE"].astype(str) == str(k_edge))

    hit = df_used.loc[m].copy()
    if len(hit) != 1:
        raise RuntimeError(f"[FATAL] fixed feature row not unique for SET={set_name}, DAG={dag}, K_EDGE={k_edge} (matched {len(hit)})")

    cols = json.loads(hit.iloc[0]["FEATURES_JSON"])
    cols = hard_check_feature_list(cols, f"fixed_features/{set_name}/{dag}/K={k_edge}")
    return cols


# -------- models (GPU try -> CPU fallback) --------
LGBM_CPU_BASE = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    verbose=-1,
)
LGBM_GPU_EXTRA = dict(device_type="gpu", gpu_platform_id=0, gpu_device_id=0)

XGB_GPU_BASE = dict(
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    n_estimators=800,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    verbosity=0,
)
XGB_CPU_BASE = dict(
    tree_method="hist",
    predictor="cpu_predictor",
    n_estimators=800,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    verbosity=0,
)

_LGBM_LOGGED = False
_XGB_LOGGED = False

def fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test, seed: int):
    global _LGBM_LOGGED
    params_gpu = dict(LGBM_CPU_BASE); params_gpu.update(LGBM_GPU_EXTRA); params_gpu["random_state"] = int(seed)
    params_cpu = dict(LGBM_CPU_BASE); params_cpu["random_state"] = int(seed)

    try:
        clf = lgb.LGBMClassifier(**params_gpu)
        clf.fit(X_train, y_train)
        if not _LGBM_LOGGED:
            print("[INFO] LightGBM GPU enabled (device_type='gpu').")
            _LGBM_LOGGED = True
    except Exception as e:
        clf = lgb.LGBMClassifier(**params_cpu)
        clf.fit(X_train, y_train)
        if not _LGBM_LOGGED:
            print(f"[WARN] LightGBM GPU failed -> CPU fallback. reason={e}")
            _LGBM_LOGGED = True

    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob, n_grid=THR_GRID)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_xgb(X_train, y_train, X_val, y_val, X_test, seed: int):
    global _XGB_LOGGED
    params_gpu = dict(XGB_GPU_BASE); params_gpu["random_state"] = int(seed)
    params_cpu = dict(XGB_CPU_BASE); params_cpu["random_state"] = int(seed)

    try:
        clf = xgb.XGBClassifier(**params_gpu)
        clf.fit(X_train, y_train)
        if not _XGB_LOGGED:
            print("[INFO] XGBoost GPU enabled (tree_method='gpu_hist').")
            _XGB_LOGGED = True
    except Exception as e:
        clf = xgb.XGBClassifier(**params_cpu)
        clf.fit(X_train, y_train)
        if not _XGB_LOGGED:
            print(f"[WARN] XGBoost GPU failed -> CPU fallback. reason={e}")
            _XGB_LOGGED = True

    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob, n_grid=THR_GRID)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr


# -------- feature manifest output --------
FEATURE_ROWS = []
def make_feature_key(payload: dict) -> str:
    s = json.dumps(payload, ensure_ascii=False, separators=(",", ":"), sort_keys=True)
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def record_features(set_name: str, dag: str, model: str, repeat: int, seed: int, cols: List[str]) -> str:
    cols = hard_check_feature_list(cols, f"record_features/{set_name}/{dag}/{model}")
    payload = {"SET": set_name, "DAG": dag, "MODEL": model, "REPEAT": int(repeat), "SEED": int(seed), "COLS": cols}
    key = make_feature_key(payload)
    FEATURE_ROWS.append({
        "FEATURE_KEY": key,
        "SET": set_name, "DAG": dag, "MODEL": model,
        "REPEAT": int(repeat), "SEED": int(seed),
        "N_FEAT": int(len(cols)),
        "FEATURES_JSON": json.dumps(cols, ensure_ascii=False),
    })
    return key


# =========================
# MAIN
# =========================
print("[LOAD] random-weight dataset splits...")
df_base_train = load_split("original", "train")
df_base_val   = load_split("original", "val")
df_base_test  = load_split("original", "test")

y_train = df_base_train[TARGET_COL].to_numpy(np.int64)
y_val   = df_base_val[TARGET_COL].to_numpy(np.int64)
y_test  = df_base_test[TARGET_COL].to_numpy(np.int64)

df_ges_train   = load_split("GES", "train")
df_ges_val     = load_split("GES", "val")
df_ges_test    = load_split("GES", "test")

df_golem_train = load_split("GOLEM", "train")
df_golem_val   = load_split("GOLEM", "val")
df_golem_test  = load_split("GOLEM", "test")

# fixed features
feat_path = find_features_used_path()
df_used = pd.read_csv(feat_path, low_memory=False)
print(f"[LOAD] features_used.csv -> {feat_path} (rows={len(df_used)})")

FIXED_F_COLS  = load_fixed_feature_list(df_used, set_name="F",  dag="GES",   k_edge=CFG_F["K_EDGE"])
FIXED_OF_COLS = load_fixed_feature_list(df_used, set_name="OF", dag="GOLEM", k_edge=CFG_OF["K_EDGE"])

print(f"[INFO] FIXED_F_COLS n={len(FIXED_F_COLS)}")
print(f"[INFO] FIXED_OF_COLS n={len(FIXED_OF_COLS)}")

# ensure columns exist
ensure_cols(df_ges_train,   FIXED_F_COLS + [TARGET_COL], "GES_train_fixed_F")
ensure_cols(df_ges_val,     FIXED_F_COLS + [TARGET_COL], "GES_val_fixed_F")
ensure_cols(df_ges_test,    FIXED_F_COLS + [TARGET_COL], "GES_test_fixed_F")

ensure_cols(df_golem_train, FIXED_OF_COLS + [TARGET_COL], "GOLEM_train_fixed_OF")
ensure_cols(df_golem_val,   FIXED_OF_COLS + [TARGET_COL], "GOLEM_val_fixed_OF")
ensure_cols(df_golem_test,  FIXED_OF_COLS + [TARGET_COL], "GOLEM_test_fixed_OF")

RESULT_ROWS = []

def run_one_repeat_F(repeat: int):
    seed = int(BASE_SEED + repeat)
    X_tr = df_ges_train[FIXED_F_COLS].to_numpy(np.float32)
    X_va = df_ges_val[FIXED_F_COLS].to_numpy(np.float32)
    X_te = df_ges_test[FIXED_F_COLS].to_numpy(np.float32)

    feature_key = record_features("F", "GES", "LightGBM", repeat, seed, FIXED_F_COLS)

    prob, thr = fit_predict_lgbm(X_tr, y_train, X_va, y_val, X_te, seed=seed)
    metrics = compute_metrics(y_test, prob, thr)

    RESULT_ROWS.append({
        "SET": "F", "DAG": "GES", "MODEL": "LightGBM",
        "K_EDGE": int(CFG_F["K_EDGE"]), "N_FEAT_TARGET": int(CFG_F["N_FEAT"]),
        "REPEAT": repeat, "SEED": seed, "THRESHOLD": float(thr),
        "FEATURE_KEY": feature_key,
        **metrics
    })

def run_one_repeat_OF(repeat: int):
    seed = int(BASE_SEED + repeat)
    X_tr = df_golem_train[FIXED_OF_COLS].to_numpy(np.float32)
    X_va = df_golem_val[FIXED_OF_COLS].to_numpy(np.float32)
    X_te = df_golem_test[FIXED_OF_COLS].to_numpy(np.float32)

    feature_key = record_features("OF", "GOLEM", "XGBoost", repeat, seed, FIXED_OF_COLS)

    prob, thr = fit_predict_xgb(X_tr, y_train, X_va, y_val, X_te, seed=seed)
    metrics = compute_metrics(y_test, prob, thr)

    RESULT_ROWS.append({
        "SET": "OF", "DAG": "GOLEM", "MODEL": "XGBoost",
        "K_EDGE": int(CFG_OF["K_EDGE"]), "N_FEAT_TARGET": int(CFG_OF["N_FEAT"]),
        "N_FEAT_ACTUAL": int(len(FIXED_OF_COLS)),
        "REPEAT": repeat, "SEED": seed, "THRESHOLD": float(thr),
        "FEATURE_KEY": feature_key,
        **metrics
    })

print("[RUN] TOP2 (F + OF) repeats: 100 each (total 200)")
t0 = time.time()
for r in range(N_REPEATS):
    run_one_repeat_F(r)
    run_one_repeat_OF(r)
    if (r + 1) % 10 == 0:
        print(f"  done repeats: {r+1}/{N_REPEATS} | elapsed={time.time()-t0:.1f}s")

df_res = pd.DataFrame(RESULT_ROWS)
df_feat = pd.DataFrame(FEATURE_ROWS)

# hard check again
s = df_feat["FEATURES_JSON"].astype(str)
bad_mask = s.str.contains("Unnamed", case=False, na=False) | s.str.contains('"index"', case=False, na=False) | s.str.contains('_index"', case=False, na=False)
if bad_mask.any():
    bad_rows = df_feat.loc[bad_mask].head(5)
    raise RuntimeError(f"[FATAL] features_used_top2_repeats contains forbidden columns. Example:\n{bad_rows}")

df_res.to_csv(OUT_RESULTS, index=False)
df_feat.to_csv(OUT_FEATURES, index=False)

print("[DONE] saved:")
print(" -", OUT_RESULTS)
print(" -", OUT_FEATURES)

display(df_res.head(10))
display(df_res.groupby(["SET","DAG","MODEL"])[["AUROC","AUPRC","F1","Brier","ECE"]].mean().reset_index())
